# [Notebook]Unihack26_Change_Head
Synthetic paired visible-change pretraining. Known masks with ignored occlusion. Object-disjoint train/selection/calibration/test. Not a physical damage benchmark.

In [ ]:
from pathlib import Path
import sys,json
SOURCES={'ml/__init__.py': '', 'ml/data/__init__.py': '', 'ml/data/labels.py': '"""Validate pair supervision and detect split leakage before feature extraction."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections import defaultdict\n\nSPLITS = ("train", "validation", "test")\nCORRESPONDENCE = {"MATCH", "WRONG_REGION", "WRONG_OBJECT", "UNKNOWN"}\nSUFFICIENCY = {"SUFFICIENT", "INSUFFICIENT", "UNKNOWN"}\nCHANGE = {"CHANGED", "UNCHANGED", "UNKNOWN"}\nSHA256 = re.compile(r"^[a-f0-9]{64}$")\nREASONS = {"BLUR", "GLARE", "OCCLUSION", "ROI_OUTSIDE_OVERLAP", "LOW_RESOLUTION",\n           "VIEW_MISMATCH", "NO_REFERENCE", "UNKNOWN"}\n\n\ndef locked_split(lineages: list[str], seed: str = "unihack26-v1") -> dict:\n    """Split a fixed, deduplicated corpus 60/20/20. Persist; never refit mid-run."""\n    if len(lineages) != len(set(lineages)) or any(not item for item in lineages):\n        raise ValueError("LINEAGE_IDS_MUST_BE_UNIQUE_AND_NONEMPTY")\n    if len(lineages) < 5:\n        raise ValueError("INSUFFICIENT_LINEAGES_FOR_THREE_SPLITS")\n    ordered = sorted(lineages, key=lambda item: hashlib.sha256(f"{seed}:{item}".encode()).digest())\n    train_end, validation_end = int(len(ordered) * 0.6), int(len(ordered) * 0.8)\n    groups = {name: split for split, names in zip(\n        SPLITS, (ordered[:train_end], ordered[train_end:validation_end], ordered[validation_end:]), strict=True)\n        for name in names}\n    payload = {"version": "lineage-split-v1", "seed": seed, "lineages": groups}\n    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)\n    return {**payload, "sha256": hashlib.sha256(canonical.encode()).hexdigest()}\n\n\ndef valid_polygons(polygons) -> bool:\n    if not isinstance(polygons, list):\n        return False\n    for polygon in polygons:\n        if not isinstance(polygon, list) or len(polygon) < 3:\n            return False\n        for point in polygon:\n            if not isinstance(point, list) or len(point) != 2:\n                return False\n            if any(isinstance(v, bool) or not isinstance(v, (int, float))\n                   or not math.isfinite(v) or not 0 <= v <= 1 for v in point):\n                return False\n        area = abs(sum(polygon[i][0] * polygon[(i + 1) % len(polygon)][1]\n                       - polygon[(i + 1) % len(polygon)][0] * polygon[i][1]\n                       for i in range(len(polygon)))) / 2\n        if area <= 1e-8:\n            return False\n    return True\n\n\ndef validate_pairs(records: list[dict], split_manifest: dict) -> dict:\n    """Return explicit failures. UNKNOWN is masked, never interpreted as a negative."""\n    errors = []\n    seen_pairs = set()\n    hashes = defaultdict(set)\n    queries = defaultdict(set)\n    mapping = split_manifest.get("lineages", {})\n    counts = defaultdict(int)\n    for index, record in enumerate(records):\n        def fail(code):\n            errors.append({"row": index, "pair_id": record.get("pair_id"), "code": code})\n\n        required = ("pair_id", "query_id", "split", "query_lineage_id", "reference_lineage_id",\n                    "query_sha256", "reference_sha256", "source_kind", "source_ref", "license_ref",\n                    "annotator_id", "correspondence", "sufficiency", "observable_change",\n                    "change_polygons", "ignore_polygons", "reason_codes")\n        if any(key not in record for key in required):\n            fail("MISSING_REQUIRED_FIELDS")\n            continue\n        text_fields = set(required) - {"change_polygons", "ignore_polygons", "reason_codes"}\n        if any(not isinstance(record[key], str) or not record[key].strip() for key in text_fields):\n            fail("INVALID_TEXT_FIELD")\n            continue\n        pair_id, split = record["pair_id"], record["split"]\n        if pair_id in seen_pairs:\n            fail("DUPLICATE_PAIR_ID")\n        seen_pairs.add(pair_id)\n        if split not in SPLITS:\n            fail("INVALID_SPLIT")\n        counts[split] += 1\n        for side in ("query", "reference"):\n            if mapping.get(record[f"{side}_lineage_id"]) != split:\n                fail(f"{side.upper()}_LINEAGE_SPLIT_MISMATCH")\n            digest = record[f"{side}_sha256"]\n            if not SHA256.fullmatch(digest):\n                fail("INVALID_IMAGE_SHA256")\n            hashes[digest].add(split)\n        queries[record["query_id"]].add(split)\n        correspondence, sufficiency, change = (record[key] for key in (\n            "correspondence", "sufficiency", "observable_change"))\n        if correspondence not in CORRESPONDENCE or sufficiency not in SUFFICIENCY or change not in CHANGE:\n            fail("INVALID_LABEL_ENUM")\n        if correspondence != "MATCH" and sufficiency == "SUFFICIENT":\n            fail("SUFFICIENT_REQUIRES_MATCH")\n        if change != "UNKNOWN" and (correspondence != "MATCH" or sufficiency != "SUFFICIENT"):\n            fail("CHANGE_REQUIRES_SUFFICIENT_MATCH")\n        if correspondence in ("MATCH", "WRONG_REGION") and record["query_lineage_id"] != record["reference_lineage_id"]:\n            fail("SAME_OBJECT_LABEL_LINEAGE_MISMATCH")\n        if correspondence == "WRONG_OBJECT" and record["query_lineage_id"] == record["reference_lineage_id"]:\n            fail("WRONG_OBJECT_REQUIRES_DIFFERENT_LINEAGE")\n        if record["source_kind"] not in {"REAL", "SYNTHETIC", "SOFTWARE_FIXTURE"}:\n            fail("INVALID_SOURCE_KIND")\n        if not valid_polygons(record["change_polygons"]) or not valid_polygons(record["ignore_polygons"]):\n            fail("INVALID_NORMALIZED_POLYGON")\n        if record["change_polygons"] and change != "CHANGED":\n            fail("CHANGE_MASK_WITHOUT_CHANGED_LABEL")\n        if not isinstance(record["reason_codes"], list) or any(not isinstance(reason, str) or reason not in REASONS for reason in record["reason_codes"]):\n            fail("INVALID_REASON_CODES")\n    for digest, splits in hashes.items():\n        if len(splits) > 1:\n            errors.append({"code": "IMAGE_HASH_SPLIT_LEAKAGE", "sha256": digest, "splits": sorted(splits)})\n    for query_id, splits in queries.items():\n        if len(splits) > 1:\n            errors.append({"code": "QUERY_SPLIT_LEAKAGE", "query_id": query_id, "splits": sorted(splits)})\n    return {"gate": "PASS" if records and not errors else "FAIL", "rows": len(records),\n            "split_counts": dict(counts), "errors": errors,\n            "license_clearance": "SEPARATE_REVIEW_REQUIRED", "near_duplicate_audit": "SEPARATE_REVIEW_REQUIRED"}\n\n\ndef supervision(record: dict) -> dict:\n    match = record["correspondence"]\n    sufficient = record["sufficiency"]\n    return {\n        "same_region_target": float(match == "MATCH"),\n        "same_region_mask": match != "UNKNOWN",\n        "sufficiency_target": float(sufficient == "SUFFICIENT"),\n        "sufficiency_mask": match == "MATCH" and sufficient != "UNKNOWN",\n        "change_mask": match == "MATCH" and sufficient == "SUFFICIENT" and record["observable_change"] != "UNKNOWN",\n    }\n', 'ml/data/carton_views.py': '"""Procedural carton-face pretraining with ground-truth planar correspondence.\n\nNo Parcel3D pixels, external textures, brand assets or user photos are used.\nThis is synthetic pretraining, NOT a physical damage or real-box benchmark.\nFaces from a box share print style; distinct faces and nuisance views are hard\nnegatives. Labels derive from generation geometry, never SIFT/model predictions.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nfrom dataclasses import dataclass\n\nimport cv2\nimport numpy as np\n\nfrom ml.data.labels import locked_split\n\nVERSION = "carton-planar-pretrain-v1"\nSIZE = 384\nROI = [[2500, 2500], [7500, 2500], [7500, 7500], [2500, 7500]]\n\n\n@dataclass\nclass View:\n    id: str\n    face: int\n    image: np.ndarray\n    transform: np.ndarray\n    visible: np.ndarray\n    condition: str\n\n\ndef face_texture(box: int, face: int, size: int = SIZE) -> np.ndarray:\n    rng = np.random.default_rng(np.random.SeedSequence([26, box, face]))\n    base = np.array([185, 150, 104]) + rng.integers(-25, 26, 3)\n    image = np.clip(base + rng.normal(0, 3, (size, size, 1)), 0, 255).astype(np.uint8)\n    ink = tuple(int(x) for x in np.array([45, 38, 30]) + (box % 7) * 3)\n    # Common carton seam, variable tape and repeated generic packing marks.\n    seam = int(rng.integers(140, 240))\n    cv2.rectangle(image, (seam, 0), (seam + 26, size), (166, 133, 90), -1)\n    for _ in range(30):\n        x, y = rng.integers(8, size - 30, 2)\n        w, h = rng.integers(4, 45, 2)\n        cv2.rectangle(image, (int(x), int(y)), (int(x+w), int(y+h)), ink, int(rng.integers(1, 3)))\n    # Neutral synthetic label; no stable face/class identifier is printed.\n    x, y = (int(v) for v in rng.integers(20, 145, 2))\n    cv2.rectangle(image, (x, y), (x+160, y+90), (230, 225, 211), -1)\n    for index in range(45):\n        thickness = int(rng.integers(1, 4))\n        cv2.line(image, (x+8+index*3, y+12), (x+8+index*3, y+52), ink, thickness)\n    code = "".join(str(v) for v in rng.integers(0, 10, 10))\n    cv2.putText(image, code, (x+8, y+76), cv2.FONT_HERSHEY_SIMPLEX, 0.45, ink, 1, cv2.LINE_AA)\n    return image\n\n\ndef render_view(texture: np.ndarray, rng, identifier: str, face: int, condition: str) -> View:\n    source = np.float32([[0, 0], [SIZE-1, 0], [SIZE-1, SIZE-1], [0, SIZE-1]])\n    destination = source + rng.uniform(-38, 38, (4, 2)).astype(np.float32)\n    if condition == "partial":\n        destination[:, 0] -= 165\n    transform = cv2.getPerspectiveTransform(source, destination)\n    image = cv2.warpPerspective(texture, transform, (SIZE, SIZE), borderValue=(73, 79, 82))\n    visible = cv2.warpPerspective(np.ones((SIZE, SIZE), np.uint8), transform, (SIZE, SIZE), flags=cv2.INTER_NEAREST)\n    gain = rng.uniform(0.65, 1.3)\n    bias = rng.uniform(-16, 16)\n    image = np.clip(image.astype(np.float32) * gain + bias, 0, 255).astype(np.uint8)\n    if condition == "blur":\n        image = cv2.GaussianBlur(image, (27, 27), 7)\n    elif condition == "occlusion":\n        cv2.rectangle(image, (95, 120), (290, 275), (75, 83, 90), -1)\n        visible[120:276, 95:291] = 0\n    elif condition == "glare":\n        cv2.ellipse(image, (200, 200), (100, 60), 20, 0, 360, (255, 255, 255), -1)\n        cv2.ellipse(visible, (200, 200), (100, 60), 20, 0, 360, 0, -1)\n    return View(identifier, face, image, transform, visible, condition)\n\n\ndef box_views(box: int):\n    rng = np.random.default_rng(np.random.SeedSequence([2626, box]))\n    textures = [face_texture(box, face) for face in range(3)]\n    references = [render_view(textures[face], rng, f"b{box}:r{face}:{view}", face,\n                              "partial" if view == 1 else "clean")\n                  for face in range(3) for view in range(2)]\n    # Rotate face/condition independently of the locked split.\n    conditions = ["clean", "blur", "occlusion", "glare"]\n    queries = [render_view(textures[(box+i) % 3], rng, f"b{box}:q{i}", (box+i) % 3, condition)\n               for i, condition in enumerate(conditions)]\n    return queries, references\n\n\ndef pair_label(query: View, reference: View):\n    same = query.face == reference.face\n    overlap = 0.0\n    if same:\n        q_to_r = reference.transform @ np.linalg.inv(query.transform)\n        available = cv2.warpPerspective(reference.visible, np.linalg.inv(q_to_r), (SIZE, SIZE), flags=cv2.INTER_NEAREST)\n        roi = np.zeros((SIZE, SIZE), np.uint8)\n        cv2.fillPoly(roi, [np.rint(np.array(ROI)/10000*(SIZE-1)).astype(np.int32)], 1)\n        overlap = float(np.sum(available * query.visible * roi) / roi.sum())\n    sufficient = same and overlap >= 0.90 and query.condition != "blur"\n    return {"same_region": float(same), "sufficient": float(sufficient),\n            "ground_truth_overlap": overlap, "label_method": "KNOWN_GENERATOR_GEOMETRY"}\n\n\ndef corpus_split(boxes: int):\n    return locked_split([f"carton:{box}" for box in range(boxes)], seed=VERSION)\n\n\ndef image_hash(image):\n    return hashlib.sha256(image.tobytes()).hexdigest()\n', 'ml/data/appearance_changes.py': '"""Self-generated visible-change pairs with nuisance/ignore masks.\n\nThis is synthetic appearance-change pretraining, not a real-damage benchmark.\nNo label claims a dent, a cause, or responsibility.\n"""\nimport cv2\nimport numpy as np\n\nfrom ml.data.carton_views import face_texture\nfrom ml.data.labels import locked_split\n\nVERSION = "carton-appearance-change-v1"\nSIZE = 128\n\n\ndef split_manifest(boxes=240):\n    return locked_split([f"appearance:{i}" for i in range(boxes)], seed=VERSION)\n\n\ndef example(box, scenario):\n    rng = np.random.default_rng(np.random.SeedSequence([262611, box, scenario]))\n    before = cv2.resize(face_texture(20000+box, box % 3), (SIZE, SIZE))\n    after = before.copy()\n    mask = np.zeros((SIZE, SIZE), np.uint8)\n    valid = np.ones_like(mask)\n    changed = scenario % 2 == 1\n    if changed:\n        center = rng.integers(25, 104, 2)\n        if box % 2:\n            points = center + rng.integers(-18, 19, (7, 2))\n            points = cv2.convexHull(points.astype(np.int32))\n            cv2.fillConvexPoly(mask, points, 1)\n        else:\n            end = center+rng.integers(-22, 23, 2)\n            cv2.line(mask, tuple(center), tuple(end), 1, int(rng.integers(2, 7)))\n        color = rng.integers(25, 80, 3) if box % 3 else rng.integers(190, 246, 3)\n        after[mask > 0] = color\n    # Continuous lighting variation across the face; no change label is added.\n    gradient = np.linspace(rng.uniform(.8, 1.1), rng.uniform(.8, 1.1), SIZE)[None, :, None]\n    after = np.clip(after.astype(np.float32)*gradient*rng.uniform(.75,1.25)+rng.uniform(-10,10),0,255).astype(np.uint8)\n    # Subpixel registration residual is a hard negative around printed edges.\n    shift = np.float32([[1,0,rng.uniform(-.65,.65)],[0,1,rng.uniform(-.65,.65)]])\n    before = cv2.warpAffine(before,shift,(SIZE,SIZE),borderMode=cv2.BORDER_REFLECT_101)\n    valid[:3]=valid[-3:]=0\n    valid[:,:3]=valid[:,-3:]=0\n    if scenario >= 2:\n        x,y=(int(v) for v in rng.integers(15,90,2))\n        cv2.rectangle(after,(x,y),(x+23,y+25),(255,255,255) if scenario == 2 else (75,80,90),-1)\n        valid[max(0,y-2):min(SIZE,y+28),max(0,x-2):min(SIZE,x+26)]=0\n    # Correct global illumination without looking at the change label.\n    corrected=before.astype(np.float32)\n    for c in range(3):\n        r,q=corrected[:,:,c][valid>0],after[:,:,c][valid>0].astype(np.float32)\n        rq,qq=np.percentile(r,[20,50,80]),np.percentile(q,[20,50,80])\n        gain=np.clip((qq[2]-qq[0])/max(rq[2]-rq[0],10),.5,2)\n        corrected[:,:,c]=corrected[:,:,c]*gain+qq[1]-rq[1]*gain\n    r=np.clip(corrected,0,255)/255\n    q=after.astype(np.float32)/255\n    features=np.concatenate([q,r,np.abs(q-r),valid[:,:,None]],axis=2).transpose(2,0,1).astype(np.float32)\n    return features, mask[None].astype(np.float32), valid[None].astype(np.float32)\n', 'ml/train/__init__.py': '', 'ml/train/change_head.py': '"""Paired small decoder with valid-pixel BCE/Dice and disjoint calibration."""\nimport hashlib\nimport json\nimport platform\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom ml.data.appearance_changes import VERSION, example, split_manifest\n\n\nclass ChangeHead(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.enc=nn.Sequential(nn.Conv2d(10,16,3,padding=1),nn.GELU(),nn.Conv2d(16,16,3,padding=1),nn.GELU())\n        self.low=nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.GELU(),nn.Conv2d(32,32,3,padding=1),nn.GELU())\n        self.dec=nn.Sequential(nn.Conv2d(48,16,3,padding=1),nn.GELU(),nn.Conv2d(16,1,1))\n\n    def forward(self,x):\n        skip=self.enc(x)\n        low=self.low(F.avg_pool2d(skip,2))\n        return self.dec(torch.cat([skip,F.interpolate(low,size=skip.shape[-2:],mode="bilinear",align_corners=False)],dim=1))\n\n\ndef loss(logits,target,valid):\n    pixel=F.binary_cross_entropy_with_logits(logits,target,reduction="none",pos_weight=torch.tensor(6.,device=logits.device))\n    bce=(pixel*valid).sum()/valid.sum().clamp_min(1)\n    p=logits.sigmoid()*valid\n    t=target*valid\n    dims=(1,2,3)\n    dice=1-(2*(p*t).sum(dims)+1)/(p.sum(dims)+t.sum(dims)+1)\n    return bce+dice.mean()\n\n\ndef measure(probabilities, target, valid, threshold):\n    predicted=(probabilities>=threshold)*valid\n    target=target*valid\n    dims=(1,2,3)\n    intersection=(predicted*target).sum(dims)\n    union=np.maximum(predicted,target).sum(dims)\n    positive=target.sum(dims)>=3\n    detected=predicted.sum(dims)>=6\n    return {"images":len(target),"positive_images":int(positive.sum()),\n        "positive_mean_iou":float(np.mean(intersection[positive]/np.maximum(union[positive],1))) if positive.any() else None,\n        "positive_pixel_recall":float(intersection.sum()/max(target.sum(),1)),\n        "positive_image_recall":float(detected[positive].mean()) if positive.any() else None,\n        "negative_images":int((~positive).sum()),"negative_false_alarms":int(detected[~positive].sum()),\n        "negative_false_alarm_rate":float(detected[~positive].mean()) if (~positive).any() else None}\n\n\ndef infer(model,x,device):\n    model.eval()\n    with torch.inference_mode():\n        return np.concatenate([model(torch.from_numpy(x[i:i+16]).to(device)).sigmoid().cpu().numpy() for i in range(0,len(x),16)])\n\n\ndef run(output,boxes=240,epochs=18):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    started=time.monotonic();torch.set_num_threads(2);cv2.setNumThreads(1)\n    device="cuda" if torch.cuda.is_available() else "cpu"\n    split=split_manifest(boxes)\n    values=[example(box,scenario) for box in range(boxes) for scenario in range(4)]\n    x,y,v=[np.stack([row[i] for row in values]) for i in range(3)]\n    del values\n    box_ids=np.repeat(np.arange(boxes),4)\n    def indices(kind):\n        return np.array([i for i,box in enumerate(box_ids) if split["lineages"][f"appearance:{box}"]==kind])\n    train,test=indices("train"),indices("test")\n    validation_boxes=sorted({int(box_ids[i]) for i in indices("validation")})\n    selection_boxes=set(validation_boxes[::2]);calibration_boxes=set(validation_boxes[1::2])\n    selection=np.flatnonzero(np.isin(box_ids,list(selection_boxes)))\n    calibration=np.flatnonzero(np.isin(box_ids,list(calibration_boxes)))\n    # Seed choice is fixed a priori; epoch selection uses selection loss only.\n    torch.manual_seed(26);rng=np.random.default_rng(26)\n    model=ChangeHead().to(device);optimizer=torch.optim.AdamW(model.parameters(),lr=.001)\n    best=float("inf");best_state=None;logs=[]\n    for epoch in range(epochs):\n        model.train();losses=[]\n        order=rng.permutation(train)\n        for start in range(0,len(order),16):\n            idx=order[start:start+16]\n            logits=model(torch.from_numpy(x[idx]).to(device))\n            value=loss(logits,torch.from_numpy(y[idx]).to(device),torch.from_numpy(v[idx]).to(device))\n            optimizer.zero_grad();value.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5);optimizer.step()\n            losses.append(float(value.detach()))\n        model.eval();selection_losses=[]\n        with torch.inference_mode():\n            for start in range(0,len(selection),16):\n                idx=selection[start:start+16]\n                selection_losses.append(float(loss(model(torch.from_numpy(x[idx]).to(device)),torch.from_numpy(y[idx]).to(device),torch.from_numpy(v[idx]).to(device))))\n        validation=float(np.mean(selection_losses))\n        logs.append({"epoch":epoch+1,"train_loss":float(np.mean(losses)),"selection_loss":validation})\n        print(json.dumps(logs[-1]),flush=True)\n        if validation<best:\n            best=validation;best_state={key:value.detach().cpu().clone() for key,value in model.state_dict().items()}\n    model.load_state_dict(best_state)\n    probabilities=infer(model,x[calibration],device)\n    candidates=[]\n    for threshold in (.3,.4,.5,.6,.7,.8,.9,.95):\n        metrics=measure(probabilities,y[calibration],v[calibration],threshold)\n        if metrics["negative_false_alarm_rate"] <= .05:\n            candidates.append((metrics["positive_mean_iou"],threshold))\n    threshold=max(candidates)[1] if candidates else 1.\n    prediction=infer(model,x[test],device)\n    metrics=measure(prediction,y[test],v[test],threshold)\n    residual=np.max(x[test,6:9],axis=1,keepdims=True)\n    residual_candidates=[]\n    for cutoff in (.05,.08,.12,.16,.20,.30):\n        m=measure(np.max(x[calibration,6:9],axis=1,keepdims=True),y[calibration],v[calibration],cutoff)\n        if m["negative_false_alarm_rate"]<=.05:residual_candidates.append((m["positive_mean_iou"],cutoff))\n    residual_threshold=max(residual_candidates)[1] if residual_candidates else 1.\n    torch.save(best_state,output/"change-head.pt")\n    probe=x[selection[:3]]\n    with torch.inference_mode(): logits=model(torch.from_numpy(probe).to(device)).cpu().numpy()\n    np.savez_compressed(output/"export-parity.npz",features=probe,logits=logits)\n    card={"model_version":"synthetic-change-head-v1","dataset":VERSION,"source_kind":"SYNTHETIC",\n        "real_image_validation":"NOT_RUN","release_gate":"HOLD_REAL_BOX_VALIDATION","boxes":boxes,"images":len(x),\n        "split":split,"selection_boxes":sorted(selection_boxes),"calibration_boxes":sorted(calibration_boxes),\n        "threshold":threshold,"selection_loss":best,"calibration_policy":"max positive IoU subject to <=5% negative alarms",\n        "test":metrics,"residual_baseline":{"threshold":residual_threshold,**measure(residual,y[test],v[test],residual_threshold)},\n        "checkpoint_sha256":hashlib.sha256((output/"change-head.pt").read_bytes()).hexdigest(),\n        "runtime":{"python":platform.python_version(),"torch":torch.__version__,"seconds":time.monotonic()-started,"device":device},\n        "limitations":["Procedural texture changes only; no physical dent diagnosis.","Generated ignore masks; no trained occlusion detector.","Subpixel alignment nuisance only; no guarantee under real nonplanar deformation.","No physical-box evaluation; cannot determine responsibility or compensation."]}\n    (output/"model-card.json").write_text(json.dumps(card,indent=2),encoding="utf-8")\n    (output/"training-log.json").write_text(json.dumps(logs,indent=2),encoding="utf-8")\n    print(json.dumps(card),flush=True)\n    return card\n'}
root=Path('/kaggle/working/_src')
for name,source in SOURCES.items():
 p=root/name
 p.parent.mkdir(parents=True,exist_ok=True)
 p.write_text(source,encoding='utf-8')
sys.path.insert(0,str(root))
from ml.train.change_head import run
run('/kaggle/working/training')
Path('/kaggle/working/training/source-hashes.json').write_text(json.dumps({'ml/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/labels.py': 'ee7341c80f127e0f5d9cfa28ad9b58bb7a3ddff12e8dec893c65802d8e963bef', 'ml/data/carton_views.py': '521bef9e87ec14ec4c546ab58face8b5696310cd449d3101b962c522c1e9b66c', 'ml/data/appearance_changes.py': '8eecd63e37156144e46be7c58317e1ad6b7fec37f5a50d95f19651d4f045291b', 'ml/train/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/train/change_head.py': 'fcf21806212f83779c3e543d8b0939b843f6ec3ddd441f287e51c38ec2420cce'},indent=2))
